# Correlación de Spearman miRNA–diana
Cruza las predicciones de psRNATarget con los DEGs de RNA-seq y calcula la correlación de Spearman entre la expresión del miRNA (counts normalizados por size-factor de DESeq2) y la expresión del gen diana (TPM por muestra).

### Inputs requeridos
- `psRNATargetJob.txt` :Output de psRNATarget
- `DEG_EcoD_vs_EndoD.csv` :DEGs EcoD vs EndoD 
- `Expression_Profile_ppe_Prunus_dulcis_Lauranne_miRNA.xlsx` :Raw counts miRNA 
- `Expression_Profile_Nonpareil_gene.xlsx` :TPM génico 
- `metadatos.xlsx` :Metadatos muestras 

### Outputs
- `tabla_spearman_mirna_diana.csv` :pares significativos (ρ < −0.4, padj < 0.05)
- `tabla_spearman_completa.csv` :todos los pares calculados

## 0. Parámetros — editar las rutas en caso de ser necesario

In [ ]:
PSRNATARGET    = "psRNATargetJob.txt"
DEG_FILE       = "DEG_EcoD_vs_EndoD.csv"
MIRNA_EXPR     = "Expression_Profile_ppe_Prunus_dulcis_Lauranne_miRNA.xlsx"
GENE_EXPR      = "Expression_Profile_Nonpareil_gene.xlsx"
METADATA       = "metadatos.xlsx"

OUT_SIG        = "tabla_spearman_mirna_diana.csv"
OUT_FULL       = "tabla_spearman_completa.csv"

RHO_THRESHOLD  = -0.4
PADJ_THRESHOLD =  0.05

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
import warnings
warnings.filterwarnings('ignore')

## 2. Metadatos

In [ ]:
meta = pd.read_excel(METADATA)
meta = meta.dropna(subset=['ID'])
meta['ID'] = meta['ID'].astype(int)
meta['Dormancy state'] = meta['Dormancy state'].str.strip()
meta = meta.drop_duplicates(subset='ID')
meta = meta[meta['ID'] <= 39].copy()
sample_ids = sorted(meta['ID'].tolist())
print(f"{len(sample_ids)} muestras")
meta[['ID','Dormancy state','Treatment']].head(10)

## 3. Expresión génica (TPM)

In [ ]:
expr = pd.read_excel(GENE_EXPR)
for i in sample_ids:
    expr[f'{i}_TPM'] = pd.to_numeric(
        expr[f'{i}_TPM'].astype(str).str.replace(',', '.'), errors='coerce')

expr_idx = {}
desc_idx = {}
for _, row in expr.iterrows():
    vals = [float(row[f'{i}_TPM']) for i in sample_ids]
    expr_idx[row['Gene_ID']] = np.array(vals)
    desc_idx[row['Gene_ID']] = str(row.get('Description', '.'))

print(f"{len(expr_idx)} genes indexados")

## 4. Expresión miRNA — counts normalizados (DESeq2 size-factor)

Se usa normalización por **size-factor de DESeq2** en lugar de RPM porque el tamaño total de la librería de sRNA varía biológicamente entre estados de dormancia (EndoD ~863k reads vs EcoD ~551k reads de media). RPM inflaría artificialmente los valores de ecodormancia. La normalización por size-factor (mediana de ratios de DESeq2) es robusta a estos cambios en la composición global de la librería.

In [ ]:
# Cargar raw counts
mirna_raw = pd.read_excel(MIRNA_EXPR)
count_cols = [c for c in mirna_raw.columns if c.endswith('_Read_Count')]
counts_df = mirna_raw[['Mature_ID'] + count_cols].copy()
counts_df.columns = ['miRNA'] + [c.replace('_Read_Count', '') for c in count_cols]
counts_df = counts_df.set_index('miRNA').T.astype(int)  # 39 muestras × N miRNAs

# Filtro: >= 10 reads en >= 3 muestras
keep = (counts_df >= 10).sum(axis=0) >= 3
counts_filt = counts_df.loc[:, keep].copy()
print(f"miRNAs antes del filtro: {counts_df.shape[1]}")
print(f"miRNAs tras filtro:      {counts_filt.shape[1]}")

In [ ]:
# Metadatos para DESeq2
meta_deseq = meta.set_index(meta['ID'].astype(str)).copy()
meta_deseq = meta_deseq.loc[counts_filt.index].copy()
meta_deseq['dormancy_state'] = pd.Categorical(
    meta_deseq['Dormancy state'],
    categories=['EndoD', 'EndoR', 'EcoD'])
meta_deseq['treatment'] = pd.Categorical(
    meta_deseq['Treatment'].fillna('Control').astype(str),
    categories=['Control', 'Asc', 'Eth'])

# Ajustar DESeq2 
inference = DefaultInference(n_cpus=4)
dds = DeseqDataSet(
    counts=counts_filt,
    metadata=meta_deseq[['dormancy_state', 'treatment']],
    design_factors=['dormancy_state', 'treatment'],
    inference=inference,
    quiet=True
)
dds.deseq2()

size_factors = dds.obs['size_factors']
print("Size factors (primeras 10 muestras):")
print(size_factors.head(10).round(3))

In [ ]:
# Counts normalizados = raw / size_factor
norm_counts = counts_filt.div(size_factors, axis=0)

# Índice Mature_ID → array de 39 valores normalizados
mirna_idx = {}
for mirna_id in norm_counts.columns:
    vals = []
    for i in sample_ids:
        sid = str(i)
        vals.append(float(norm_counts.loc[sid, mirna_id]) if sid in norm_counts.index else np.nan)
    mirna_idx[mirna_id] = np.array(vals)

print(f"{len(mirna_idx)} miRNAs indexados con counts normalizados")

## 5. DEGs

In [ ]:
deg = pd.read_csv(DEG_FILE)
deg_map = deg.set_index('Gene_ID').to_dict('index')
deg_ids = set(deg[deg['regulation'].isin(['UP', 'DOWN'])]['Gene_ID'])
print(f"{len(deg_ids)} DEGs (UP + DOWN)")
deg['regulation'].value_counts()

## 6. psRNATarget — pares miRNA–diana

El campo `Target_Acc.` tiene formato `lcl|CM037990.1_cds_KAI5341428.1_20287`; el número final es el ID numérico del gen `PRUDU20287`.

In [ ]:
psrna = pd.read_csv(PSRNATARGET, sep='\t', comment='#')
psrna.columns = psrna.columns.str.strip()

psrna['PRUDU'] = psrna['Target_Acc.'].str.extract(
    r'_cds_KAI5\d+\.\d+_(\d+)').apply(
    lambda x: f"PRUDU{x[0]}" if pd.notna(x[0]) else np.nan, axis=1)

psrna_deg = psrna[psrna['PRUDU'].isin(deg_ids)].copy()
psrna_deg_uniq = psrna_deg.sort_values('Expectation').drop_duplicates(
    subset=['miRNA_Acc.', 'PRUDU'])

print(f"{len(psrna_deg_uniq)} pares únicos miRNA–PRUDU con diana DEG")
psrna_deg_uniq.head()

## 7. Calcular correlación de Spearman

In [ ]:
results = []

for _, row in psrna_deg_uniq.iterrows():
    mirna_id = row['miRNA_Acc.']
    prudu    = row['PRUDU']

    if mirna_id not in mirna_idx or prudu not in expr_idx:
        continue

    mirna_vec = mirna_idx[mirna_id]
    gene_vec  = expr_idx[prudu]

    mask = ~(np.isnan(mirna_vec) | np.isnan(gene_vec))
    if mask.sum() < 10:
        continue

    rho, pval = spearmanr(mirna_vec[mask], gene_vec[mask])
    gene_info = deg_map.get(prudu, {})

    results.append({
        'miRNA'          : mirna_id,
        'PRUDU'          : prudu,
        'Descripción'    : desc_idx.get(prudu, '.')[:60],
        'Mecanismo'      : row['Inhibition'],
        'Expectation'    : round(row['Expectation'], 2),
        'LFC_gen'        : round(gene_info.get('log2FoldChange', np.nan), 3),
        'Regulación_gen' : gene_info.get('regulation', ''),
        'padj_gen'       : gene_info.get('padj', np.nan),
        'rho_Spearman'   : round(rho, 3),
        'p_Spearman'     : round(pval, 4),
        'n_muestras'     : int(mask.sum()),
    })

df = pd.DataFrame(results)
print(f"{len(df)} pares calculados")
df.head()

## 8. Corrección FDR (Benjamini-Hochberg)

In [ ]:
_, pvals_fdr, _, _ = multipletests(df['p_Spearman'], method='fdr_bh')
df['padj_Spearman'] = np.round(pvals_fdr, 4)

print(f"Pares con p_Spearman < 0.05:   {(df['p_Spearman'] < 0.05).sum()}")
print(f"Pares con padj_Spearman < 0.05: {(df['padj_Spearman'] < 0.05).sum()}")

## 9. Colapsar familias miRNA

In [ ]:
df['miRNA_familia'] = df['miRNA'].str.replace(
    r'(ppe-miR\d+[a-z]?(?:\d+[a-z]?)?)[-_][a-z](?:$|-)', r'\1', regex=True)
df['miRNA_familia'] = df['miRNA_familia'].str.replace(
    r'(ppe-miR\d+)-[a-z]$', r'\1', regex=True)

df_best = df.sort_values('rho_Spearman').drop_duplicates(
    subset=['miRNA_familia', 'PRUDU'])

print(f"Pares únicos tras colapsar familias: {len(df_best)}")

## 10. Filtrar anti-correlaciones significativas

In [ ]:
df_sig = df_best[
    (df_best['rho_Spearman'] < RHO_THRESHOLD) &
    (df_best['padj_Spearman'] < PADJ_THRESHOLD)
].copy().sort_values('rho_Spearman')

print(f"Pares significativos (ρ < {RHO_THRESHOLD}, padj < {PADJ_THRESHOLD}): {len(df_sig)}")
df_sig[['miRNA_familia', 'PRUDU', 'Descripción', 'LFC_gen', 'rho_Spearman', 'padj_Spearman']]

## 11. Guardar resultados

In [ ]:
df_sig.to_csv(OUT_SIG, index=False, encoding='utf-8-sig')
df.sort_values('rho_Spearman').to_csv(OUT_FULL, index=False, encoding='utf-8-sig')

print(f"→ {OUT_SIG}  ({len(df_sig)} pares significativos)")
print(f"→ {OUT_FULL}  ({len(df)} pares totales)")